# 1 — Archiving

**RadDB** turns xarray **DataTree** radar volumes into a compact, queryable Parquet
archive. It is *network-agnostic*: any DataTree with the standard
[xradar](https://docs.openradarscience.org/projects/xradar/) coordinate layout —
MeteoSwiss, NEXRAD, OPERA — can be archived. No `pyart` is needed in the core.

This notebook covers:

1. Looking at what data you have, before archiving it
2. The **CRS contract** — the one thing you must get right
3. Archiving a volume
4. What lands on disk, and why it is laid out that way

---
## How the archive is stored

A radar is stored as **one static LUT** (per-gate geometry, computed once) plus
**one Parquet file per volume** (the moments), linked by an integer `gate_id`:

```
{archive_dir}/{radar}/LUT/{radar}_LUT.parquet          # gate centroids
{archive_dir}/{radar}/LUT/{radar}_h_plane_LUT.parquet  # horizontal faces (PPI)
{archive_dir}/{radar}/LUT/{radar}_v_plane_LUT.parquet  # vertical faces  (RHI)
{archive_dir}/{radar}/LUT/{radar}_corners_LUT.parquet  # 3-D gate corners
{archive_dir}/{radar}/LUT/{radar}_info.yaml            # site, CRS, scan geometry
{archive_dir}/{radar}/{YYYY}/{MM}/{DD}/{radar}_{YYYYMMDD}_{HHMMSS}_POL.parquet
```

The geometry is stored **once**, not once per volume — which is what keeps the
archive small. Gates with no echo are dropped at archive time (`DBZH > 0` by
default).

In [1]:
import os
from pathlib import Path

# --------------------------------------------------------------------------
# CONFIGURATION — point these at your own data
# --------------------------------------------------------------------------
# RadDB is network-agnostic: any xarray DataTree with the standard xradar
# layout works.  These tutorials use two MeteoSwiss volumes and two NEXRAD
# volumes stored as Zarr.  Set the environment variables, or edit the paths.

MCH_DIR    = Path(os.environ.get("RADDB_DATATREE_DIR", "~/data/RADAR/MCH_datatree")).expanduser()
NEXRAD_DIR = Path(os.environ.get("RADDB_NEXRAD_DIR",   "~/data/RADAR/NEXRAD_datatree")).expanduser()

# Where the archive is written.  Anywhere you like — it is just a directory.
ARCHIVE_DIR = Path(os.environ.get("RADDB_TUTORIAL_ARCHIVE",
                                  Path(os.environ.get("TMPDIR", "/tmp")) / "raddb_tutorial_archive"))

print("MCH DataTrees   :", MCH_DIR)
print("NEXRAD DataTrees:", NEXRAD_DIR)
print("Archive         :", ARCHIVE_DIR)


MCH DataTrees   : /data/RADAR/MCH_datatree
NEXRAD DataTrees: /data/RADAR/NEXRAD_datatree
Archive         : /tmp/raddb_tutorial_archive


In [2]:
import warnings
warnings.filterwarnings("ignore")

import raddb

print("raddb", raddb.__version__)

raddb 0.1.dev5+gde6070734.d20260323


## 1. What do I have?

`inventory()` answers "what is on disk?" for both sides of the workflow. Pointed at
a directory of DataTree files it reports the **input** side, grouping by the radar
name it reads from each filename prefix.

In [3]:
db = raddb.RadDB()
db.inventory(datatree_dir=MCH_DIR)

RadDB inventory — DataTree files on disk (not archived yet)
  directory : /data/RADAR/MCH_datatree
  files     : 2
  radars    : L, W  (from the filename prefix)
  time range: 2024-07-19 13:50:00 .. 2024-08-26 02:50:00
------------------------------------------------------------------------------
  radar     files  time range                                           size
  L             1  2024-08-26 02:50:00                               20.9 MB
  W             1  2024-07-19 13:50:00                               21.5 MB
------------------------------------------------------------------------------
  archive with: db.archive(datatree_dir='/data/RADAR/MCH_datatree')


In [4]:
# `detailed=True` adds a per-day breakdown and flags any radar name RadDB
# cannot use (names must be 1-4 characters from [0-9A-Z]).
db.inventory(datatree_dir=NEXRAD_DIR, detailed=True)

RadDB inventory — DataTree files on disk (not archived yet)
  directory : /data/RADAR/NEXRAD_datatree
  files     : 2
  radars    : KTLX  (from the filename prefix)
  time range: 2013-05-20 19:51:11 .. 2013-05-20 19:55:27
------------------------------------------------------------------------------
  radar     files  time range                                           size
  KTLX          2  2013-05-20 19:51:11 .. 2013-05-20 19:55:27        14.6 MB
      2013-05-20      2 volume(s)  19:51:11 .. 19:55:27
------------------------------------------------------------------------------
  archive with: db.archive(datatree_dir='/data/RADAR/NEXRAD_datatree')


## 2. The CRS contract

**A projection is mandatory to write an archive, and never needed to read one.**

There is no default, because a wrong projection is *silently* wrong: EPSG:2056
(Swiss LV95) used outside Switzerland mis-measures distance by ~20%, and the
resulting crops still look perfectly normal.

RadDB validates by **measurement, not by metadata**: it projects a 100 km geodesic
in eight directions around the radar site and compares against the truth.

In [1]:
from raddb.lut import suggest_crs, crs_distance_error

# suggest_crs returns the UTM zone for a site — a safe starting point anywhere.
print("CH   :", suggest_crs(6.99, 46.84))
print("Oklahoma, US  :", suggest_crs(-97.28, 35.33))

CH   : 32632
Oklahoma, US  : 32614


In [6]:
# Why metadata is not enough. EPSG:3857 (Web Mercator) claims the whole world.
for epsg, site, where in [(2056, (6.99, 46.84), "LV95 in Switzerland"),
                          (2056, (-97.28, 35.33), "LV95 in Oklahoma"),
                          (32614, (-97.28, 35.33), "UTM 14N in Oklahoma"),
                          (3857, (6.99, 46.84), "Web Mercator in Switzerland")]:
    err = crs_distance_error(epsg, *site)
    verdict = "REFUSED" if err > 1.0 else ("warn" if err > 0.1 else "accepted")
    print(f"  EPSG:{epsg:<6} {where:<26} {err:6.2f}%   {verdict}")

  EPSG:2056   LV95 in Switzerland          0.01%   accepted
  EPSG:2056   LV95 in Oklahoma            20.07%   REFUSED
  EPSG:32614  UTM 14N in Oklahoma          0.03%   accepted
  EPSG:3857   Web Mercator in Switzerland  47.62%   REFUSED


A CRS that distorts by more than 1% is **refused**, and the error names a
replacement so you are never left guessing:

In [7]:
try:
    raddb.RadDB(archive_dir=ARCHIVE_DIR / "_bad", crs=2056).archive(
        datatree=raddb.open_any_datatree(sorted(NEXRAD_DIR.glob("*.zarr"))[0]),
        radar="KTLX",
    )
except ValueError as exc:
    print("ValueError:", exc)

ValueError: EPSG:2056 (CH1903+ / LV95), valid for Liechtenstein; Switzerland. distorts distance by 20.1% at radar KTLX (-97.2775, 35.3331) — gate geometry, crops and cross-sections would all be wrong by that much. Suggested for this site: EPSG:32614.


## 3. Archiving

`archive()` takes either a directory of DataTree files or an in-memory DataTree.
The LUT is generated automatically from the first volume of each radar.

In [8]:
db = raddb.RadDB(archive_dir=ARCHIVE_DIR, crs=2056)   # 2056 = CH1903+/LV95
result = db.archive(datatree_dir=MCH_DIR)
result

RadDB archive
  archive_dir : /tmp/raddb_tutorial_archive
  crs         : 2056
  radars      : ['L', 'W']
  filter      : keep DBZH > 0.0
  volumes     : 2 archived, 0 failed
  elapsed     : 8s


{'n_archived': 2, 'n_failed': 0, 'radars': ['L', 'W']}

Because RadDB is network-agnostic, the same call archives NEXRAD — you only
change the projection to one valid where that radar actually is. Both radars live
side by side in the same archive.

In [9]:
db_us = raddb.RadDB(archive_dir=ARCHIVE_DIR, crs=32614)   # UTM zone 14N
db_us.archive(datatree_dir=NEXRAD_DIR)

RadDB archive
  archive_dir : /tmp/raddb_tutorial_archive
  crs         : 32614
  radars      : ['KTLX']
  filter      : keep DBZH > 0.0
  volumes     : 2 archived, 0 failed
  elapsed     : 23s


{'n_archived': 2, 'n_failed': 0, 'radars': ['KTLX']}

### Archiving from memory

If you already hold a DataTree — straight out of your own converter — skip the
disk round-trip:

```python
dt = my_converter(raw_file)                 # -> xarray DataTree
db.archive(datatree=dt, radar="A")

db.archive(datatree=[dt1, dt2, dt3], radar="A")        # several volumes
db.archive(datatree={"A": [dt_a], "W": [dt_w]})        # several radars
```

`archive()` reports per-volume failures rather than raising, so one bad volume
never takes down a long batch. A rejected CRS is the exception — that aborts.

## 4. What landed on disk

In [10]:
db = raddb.RadDB(archive_dir=ARCHIVE_DIR)
print("radars in the archive:", db.list_radars())
db.inventory()

radars in the archive: ['KTLX', 'L', 'W']
RadDB inventory — archived data
  archive_dir : /tmp/raddb_tutorial_archive
  radars      : KTLX, L, W
  volumes     : 4
  time range  : 2013-05-20 19:51:11 .. 2024-08-26 02:45:09
------------------------------------------------------------------------------
  radar   volumes  time range                                           size
  KTLX          2  2013-05-20 19:51:11 .. 2013-05-20 19:55:27        17.0 MB
  L             1  2024-08-26 02:45:09                                5.4 MB
  W             1  2024-07-19 13:45:06                                5.1 MB
------------------------------------------------------------------------------
  load with   : db.open(radars=..., time_period=(start, end))


In [11]:
for p in sorted((ARCHIVE_DIR / "L" / "LUT").iterdir()):
    print(f"  {p.name:<28} {p.stat().st_size / 1e6:8.2f} MB")

  L_LUT.parquet                   63.01 MB
  L_corners_LUT.parquet           18.40 MB
  L_h_plane_LUT.parquet           19.78 MB
  L_info.yaml                      0.08 MB
  L_v_plane_LUT.parquet            0.40 MB


### The `gate_id` — how a volume finds its geometry

One int64 per gate links a row of moments to its row of geometry:

```
gate_id = radar_code * 10^12 + sweep * 10^10 + azimuth*10 * 10^6 + range_m
```

It is decimal so you can read it by eye. `radar_code` is the base-36 value of the
zero-padded 4-character radar name (`"L"` → `000L` → 21, `"KTLX"` → 971493), which
allows **1,679,616 radars** and means an archive is self-describing — the radar
name can be recovered from the integers alone, with no registry file.

In [12]:
from raddb import encode_radar_code, decode_radar_code, decode_gate_radars

for name in ["A", "L", "KTLX"]:
    print(f"  {name:<5} -> code {encode_radar_code(name):>7} -> {decode_radar_code(encode_radar_code(name))}")

lut = db.get_lut("L")
print("\nLUT:", lut.shape)
print(lut.head(3).select(["gate_id", "sweep", "azimuth", "range", "latitude", "longitude", "altitude"]))

  A     -> code      10 -> A
  L     -> code      21 -> L
  KTLX  -> code  971493 -> KTLX

LUT: (1724400, 13)
shape: (3, 7)
┌────────────────┬───────┬─────────┬─────────────┬───────────┬───────────┬─────────────┐
│ gate_id        ┆ sweep ┆ azimuth ┆ range       ┆ latitude  ┆ longitude ┆ altitude    │
│ ---            ┆ ---   ┆ ---     ┆ ---         ┆ ---       ┆ ---       ┆ ---         │
│ i64            ┆ i32   ┆ f64     ┆ f32         ┆ f64       ┆ f64       ┆ f64         │
╞════════════════╪═══════╪═════════╪═════════════╪═══════════╪═══════════╪═════════════╡
│ 21010005000249 ┆ 1     ┆ 0.5     ┆ 249.999008  ┆ 46.043008 ┆ 8.833245  ┆ 1625.164775 │
│ 21010005000749 ┆ 1     ┆ 0.5     ┆ 749.997009  ┆ 46.047505 ┆ 8.833301  ┆ 1623.516397 │
│ 21010005001249 ┆ 1     ┆ 0.5     ┆ 1249.994995 ┆ 46.052001 ┆ 8.833358  ┆ 1621.89745  │
└────────────────┴───────┴─────────┴─────────────┴───────────┴───────────┴─────────────┘


In [13]:
# The radars a set of gate_ids spans, decoded from the integers alone
import polars as pl

sample = pl.concat([db.get_lut("L").head(2), db.get_lut("KTLX").head(2)], how="diagonal")
print(decode_gate_radars(sample["gate_id"].to_numpy()))

['KTLX', 'L']


### The site metadata

`info.yaml` records everything needed to reconstruct the geometry — including the
CRS that was validated at archive time, and the radar's **scan strategy**.

In [14]:
info = db.get_radar_info("L")
for k in ["radar", "latitude", "longitude", "altitude", "crs", "ke",
          "beamwidth_deg", "n_sweeps", "n_gates", "gate_id_version"]:
    print(f"  {k:<16} {info[k]}")

  radar            L
  latitude         46.0407600402832
  longitude        8.833216667175293
  altitude         1626.0
  crs              {'epsg': 2056, 'columns': ['x_2056', 'y_2056']}
  ke               1.3333333333333333
  beamwidth_deg    1.0
  n_sweeps         20
  n_gates          1724400
  gate_id_version  2


### One detail worth knowing: the nominal azimuth grid

An antenna reports **where it actually pointed**, which drifts a few hundredths of
a degree every rotation. Since `gate_id` resolves azimuth to 0.1°, a drifting ray
would land in a different bin on every volume and its gates would match no LUT row.

So the LUT stores the radar's *scan strategy* — `360 / n_rays` spacing at the
measured offset — and every volume's rays are snapped onto it. This is derived
per sweep from the ray count, so it gives 1.0° for Rad4Alp and 0.5° for NEXRAD
super-resolution sweeps automatically, with nothing to configure.

In [15]:
sweep1 = db.get_radar_info("L")["sweeps"][1]
print("rays in sweep 1 :", sweep1["n_azimuths"])
print("azimuths (x10)  :", sweep1["azimuths"][:8], "...")
print("i.e. degrees    :", [a / 10 for a in sweep1["azimuths"][:8]], "...")

rays in sweep 1 : 360
azimuths (x10)  : [5, 15, 25, 35, 45, 55, 65, 75] ...
i.e. degrees    : [0.5, 1.5, 2.5, 3.5, 4.5, 5.5, 6.5, 7.5] ...


In [16]:
# Every volume joins its LUT completely — nothing is silently dropped.
lut_ids = db.get_lut("L").select("gate_id")
pol = db.open(radars="L").data
matched = pol.join(lut_ids, on="gate_id", how="semi").height
print(f"{matched:,} of {pol.height:,} gates join the LUT  ({100 * matched / pol.height:.1f}%)")

347,449 of 347,449 gates join the LUT  (100.0%)


---
## Recap

```python
db = raddb.RadDB(archive_dir=..., crs=2056)   # CRS mandatory to write
db.inventory(datatree_dir=...)                # what do I have?
db.archive(datatree_dir=...)                  # or datatree=dt, radar="A"
db.list_radars(); db.get_lut("L"); db.get_radar_info("L")
```

**Next:** [2 — Opening and filtering](02_opening_and_filtering.ipynb)